<div dir="rtl" style="text-align: right;">

#  ColBERT چندزبانه و فارسی با Jina-ColBERT-v2

<div align="center">
  <img src="images/weaviate.png" alt="ColBERT Farsi" width="10%">
</div>

در نوت‌بوک قبلی (`colbert.ipynb`) با ColBERT و بازیابی multi-vector آشنا شدید، اما از مدل کلاسیک `colbert-ir/colbertv2.0` استفاده کردیم که **فقط انگلیسی** است. اگر همان مدل را روی متن فارسی امتحان کنید، نتیجه‌ی قابل‌اعتمادی نمی‌گیرید.

در این نوت‌بوک از **jina-colbert-v2** استفاده می‌کنیم؛ یک مدل ColBERT چندزبانه که ده‌ها زبان از جمله **فارسی** را پشتیبانی می‌کند. کل روند (تعریف Collection، افزودن داده، جستجو) دقیقاً مثل نوت‌بوک قبلی است؛ فقط مدل embedding عوض می‌شود.

بزن بریم!

</div>

In [1]:
from IPython.display import HTML

# راست‌چین کردن سلول‌های متنی (Markdown) در این نوت‌بوک
HTML("""
<style>
  .jp-RenderedMarkdown, .rendered_html, .text_cell_render {
    direction: rtl;
    text-align: right;
  }
</style>
""")


<div dir="rtl" style="text-align: right;">

# فهرست مطالب
- [ ۱ - چرا Jina-ColBERT-v2؟](#1)
  - [ ۱.۱ بارگذاری کتابخانه‌ها و کلاینت](#1-1)
- [ ۲ - دیتاست فارسی نمونه](#2)
- [ ۳ - بردارسازی و ساخت Collection](#3)
  - [ ۳.۱ بارگذاری مدل](#3-1)
  - [ ۳.۲ بردارسازی اسناد فارسی](#3-2)
  - [ ۳.۳ ساخت Collection و افزودن داده](#3-3)
- [ ۴ - جستجوی فارسی با ColBERT](#4)
  - [ ۴.۱ کوئری فارسی](#4-1)
  - [ ۴.۲ نمونه‌ی دوم: کوئری با اصطلاح نادر/تخصصی](#4-2)
- [ ۵ - نکته‌ی مهم درباره‌ی ابعاد (Matryoshka)](#5)
- [ ۶ - جمع‌بندی](#6)

</div>

<div dir="rtl" style="text-align: right;">

<a id='1'></a>
## ۱ - چرا Jina-ColBERT-v2؟
---

`jina-colbert-v2` یک مدل late-interaction چندزبانه است که:

- **۸۹ تا ۹۴ زبان** از جمله فارسی (`fa`) را پوشش می‌دهد
- روی بنچمارک چندزبانه‌ی MIRACL به‌طور محسوسی از BM25 بهتر عمل می‌کند
- از **Matryoshka embeddings** پشتیبانی می‌کند، یعنی می‌توانید بین ابعاد ۱۲۸، ۹۶ یا ۶۴ یکی را انتخاب کنید (تعادل بین دقت و حجم ذخیره‌سازی)
- طول context تا ۸۱۹۲ توکن را پشتیبانی می‌کند

مثل قبل، برای بردارسازی از `fastembed` استفاده می‌کنیم که این مدل را به‌صورت ONNX (بدون نیاز به GPU) در دسترس می‌گذارد.

<a id='1-1'></a>
### ۱.۱ بارگذاری کتابخانه‌ها و کلاینت

pip install weaviate-client fastembed tqdm

</div>

In [2]:
from weaviate.classes.config import Configure, Property, DataType
from weaviate.classes.query import MetadataQuery
from weaviate.util import generate_uuid5
from fastembed import LateInteractionTextEmbedding
from tqdm import tqdm
import weaviate

from utils import (
    suppress_subprocess_output,
    print_object_properties,
)

with suppress_subprocess_output():
    client = weaviate.connect_to_local()


<div dir="rtl" style="text-align: right;">

<a id='2'></a>
## ۲ - دیتاست فارسی نمونه
---

بیایید همان روحیه‌ی دیتاست گردشگری آزمایشگاه اول را این‌بار به فارسی بسازیم؛ چند جاذبه‌ی گردشگری ایران همراه با توضیح کوتاه.

</div>

In [3]:
data = [
    {
        "place": "تخت جمشید",
        "state": "فارس",
        "description": "مجموعه‌ای باستانی از دوران هخامنشیان با کاخ‌ها و ستون‌های حجاری‌شده که نمادی از معماری ایران باستان است.",
        "budget": "متوسط",
    },
    {
        "place": "میدان نقش جهان",
        "state": "اصفهان",
        "description": "یکی از بزرگ‌ترین میدان‌های تاریخی جهان، با مسجد شاه، کاخ عالی‌قاپو و بازار سنتی اصفهان.",
        "budget": "ارزان",
    },
    {
        "place": "دریاچه ارومیه",
        "state": "آذربایجان غربی",
        "description": "یکی از بزرگ‌ترین دریاچه‌های آب شور جهان، مقصدی برای طبیعت‌گردی و مشاهده‌ی پرندگان مهاجر.",
        "budget": "ارزان",
    },
    {
        "place": "برج آزادی",
        "state": "تهران",
        "description": "نماد شهر تهران با معماری ترکیبی از سبک‌های ساسانی و اسلامی، شامل موزه و نمایشگاه دائمی.",
        "budget": "ارزان",
    },
    {
        "place": "کویر مرنجاب",
        "state": "اصفهان",
        "description": "یک کویر بکر با تپه‌های شنی و آسمان شب بی‌نظیر، مناسب برای اقامت در کمپ و شترسواری.",
        "budget": "گران",
    },
    {
        "place": "خلیج‌فارس - جزیره کیش",
        "state": "هرمزگان",
        "description": "جزیره‌ای گرمسیری با سواحل مرجانی، مراکز خرید بزرگ و امکانات تفریحی برای گردشگران.",
        "budget": "گران",
    },
]

for d in data:
    print(d["place"], "-", d["description"][:40], "...")


تخت جمشید - مجموعه‌ای باستانی از دوران هخامنشیان با  ...
میدان نقش جهان - یکی از بزرگ‌ترین میدان‌های تاریخی جهان،  ...
دریاچه ارومیه - یکی از بزرگ‌ترین دریاچه‌های آب شور جهان، ...
برج آزادی - نماد شهر تهران با معماری ترکیبی از سبک‌ه ...
کویر مرنجاب - یک کویر بکر با تپه‌های شنی و آسمان شب بی ...
خلیج‌فارس - جزیره کیش - جزیره‌ای گرمسیری با سواحل مرجانی، مراکز  ...


<div dir="rtl" style="text-align: right;">

<a id='3'></a>
## ۳ - بردارسازی و ساخت Collection
---

<a id='3-1'></a>
### ۳.۱ بارگذاری مدل

اولین اجرا مدل را دانلود می‌کند. توجه: چون این مدل بزرگ‌تر و چندزبانه است، دانلود و بارگذاری اولیه ممکن است از مدل انگلیسی نوت‌بوک قبلی کمی بیشتر طول بکشد.

</div>

In [9]:
colbert_fa_model = LateInteractionTextEmbedding(model_name="jinaai/jina-colbert-v2")

<div dir="rtl" style="text-align: right;">

<a id='3-2'></a>
### ۳.۲ بردارسازی اسناد فارسی

مثل قبل، فیلدهای متنی مرتبط را به هم می‌چسبانیم و بردارسازی می‌کنیم.

</div>

In [5]:
texts = [f"{d['place']} ({d['state']}): {d['description']}" for d in data]

doc_vectors = list(colbert_fa_model.embed(texts))

print("تعداد اسناد:", len(doc_vectors))
print("شکل بردار سند اول (تعداد توکن، ابعاد):", doc_vectors[0].shape)


تعداد اسناد: 6
شکل بردار سند اول (تعداد توکن، ابعاد): (38, 128)


<div dir="rtl" style="text-align: right;">

<a id='3-3'></a>
### ۳.۳ ساخت Collection و افزودن داده

دقیقاً همان پیکربندی `self_provided` نوت‌بوک قبلی، فقط با نام Collection جدید.

</div>

In [6]:
COLLECTION_NAME = "colbert_farsi_demo"

if client.collections.exists(COLLECTION_NAME):
    client.collections.delete(COLLECTION_NAME)

collection = client.collections.create(
    name=COLLECTION_NAME,
    vector_config=[
        Configure.MultiVectors.self_provided(name="colbert"),
    ],
    properties=[
        Property(name="place", data_type=DataType.TEXT),
        Property(name="state", data_type=DataType.TEXT),
        Property(name="description", data_type=DataType.TEXT),
        Property(name="budget", data_type=DataType.TEXT),
    ],
)

with collection.batch.fixed_size(batch_size=1, concurrent_requests=1) as batch:
    for document, vec in tqdm(list(zip(data, doc_vectors))):
        uuid = generate_uuid5(document)
        batch.add_object(
            properties=document,
            uuid=uuid,
            vector={"colbert": vec.tolist()},
        )

print(len(collection))


100%|████████████████████████████████████████████████████████████████████████████████████| 6/6 [00:00<00:00, 36.67it/s]


6


<div dir="rtl" style="text-align: right;">

<a id='4'></a>
## ۴ - جستجوی فارسی با ColBERT
---

<a id='4-1'></a>
### ۴.۱ کوئری فارسی

مثل نوت‌بوک قبلی، برای بردارسازی کوئری از `query_embed` استفاده می‌کنیم (نه `embed`).

</div>

In [7]:
query = "جایی می‌خواهم بروم که کویر و آسمان پرستاره داشته باشد"

query_vector = list(colbert_fa_model.query_embed(query))[0]

result = collection.query.near_vector(
    near_vector={"colbert": query_vector.tolist()},
    target_vector="colbert",
    limit=3,
    return_metadata=MetadataQuery(distance=True),
)

for obj in result.objects:
    print(f"فاصله: {obj.metadata.distance:.4f}")
    print_object_properties(obj.properties)


فاصله: -17.1216
description: یک کویر بکر با تپه‌های شنی و آسمان شب بی‌نظیر، مناسب برای اقامت در کمپ و شترسواری.
budget: گران
place: کویر مرنجاب
state: اصفهان

فاصله: -15.4283
state: آذربایجان غربی
description: یکی از بزرگ‌ترین دریاچه‌های آب شور جهان، مقصدی برای طبیعت‌گردی و مشاهده‌ی پرندگان مهاجر.
place: دریاچه ارومیه
budget: ارزان

فاصله: -15.1549
state: اصفهان
description: یکی از بزرگ‌ترین میدان‌های تاریخی جهان، با مسجد شاه، کاخ عالی‌قاپو و بازار سنتی اصفهان.
place: میدان نقش جهان
budget: ارزان



<div dir="rtl" style="text-align: right;">

<a id='4-2'></a>
### ۴.۲ نمونه‌ی دوم: کوئری با اصطلاح نادر/تخصصی

بیایید یک کوئری با اسم خاص تاریخی امتحان کنیم تا قدرت تطبیق توکن‌به‌توکن ColBERT را ببینیم.

</div>

In [8]:
query2 = "آثار باقی‌مانده از دوران هخامنشیان کجاست؟"

query_vector2 = list(colbert_fa_model.query_embed(query2))[0]

result2 = collection.query.near_vector(
    near_vector={"colbert": query_vector2.tolist()},
    target_vector="colbert",
    limit=2,
    return_metadata=MetadataQuery(distance=True),
)

for obj in result2.objects:
    print(f"فاصله: {obj.metadata.distance:.4f}")
    print_object_properties(obj.properties)


فاصله: -20.0332
state: فارس
description: مجموعه‌ای باستانی از دوران هخامنشیان با کاخ‌ها و ستون‌های حجاری‌شده که نمادی از معماری ایران باستان است.
place: تخت جمشید
budget: متوسط

فاصله: -18.4681
budget: ارزان
state: اصفهان
place: میدان نقش جهان
description: یکی از بزرگ‌ترین میدان‌های تاریخی جهان، با مسجد شاه، کاخ عالی‌قاپو و بازار سنتی اصفهان.



<div dir="rtl" style="text-align: right;">

<a id='5'></a>
## ۵ - نکته‌ی مهم درباره‌ی ابعاد (Matryoshka)
---

مدل `jinaai/jina-colbert-v2` سه نسخه دارد که در ابعاد بردار هر توکن فرق دارند:

- `jinaai/jina-colbert-v2` → ۱۲۸ بعد (بیشترین دقت)
- `jinaai/jina-colbert-v2-96` → ۹۶ بعد
- `jinaai/jina-colbert-v2-64` → ۶۴ بعد (کمترین حجم ذخیره‌سازی)

چون هر سند در ColBERT به‌جای یک بردار، **صدها بردار** دارد (یکی به‌ازای هر توکن)، انتخاب بعد کوچک‌تر می‌تواند تفاوت محسوسی در حجم ایندکس ایجاد کند. اگر دیتاست بزرگی دارید و محدودیت حافظه دارید، امتحان نسخه‌های ۹۶ یا ۶۴ بعدی منطقی است؛ کافیست نام مدل را در `LateInteractionTextEmbedding(model_name=...)` عوض کنید.

</div>

<div dir="rtl" style="text-align: right;">

<a id='6'></a>
## ۶ - جمع‌بندی
---

در این نوت‌بوک یاد گرفتید:
- چرا ColBERT اصلی برای فارسی مناسب نیست و به یک نسخه‌ی چندزبانه نیاز داریم
- چطور با `jina-colbert-v2` روی متن فارسی بردارسازی و جستجو کنید
- گزینه‌های Matryoshka برای کاهش حجم ذخیره‌سازی

در نوت‌بوک بعدی (`colbert_multimodal.ipynb`) یک قدم جلوتر می‌رویم و به‌جای متن، روی **تصویر** با ColPali کار می‌کنیم.

</div>

In [ ]:
# فراموش نکنید کلاینت را ببندید!
client.close()
